In [ ]:
import numpy as np
from pathlib import Path
from typing import List
import zarr

from sample_db import SampleDB

In [ ]:

class SumTree(object):
    """SumTree datastructure for storing priority values in binary heap like structure for fast sampling"""

    def __init__(self, num_elements):
        assert num_elements != 0 and (
            (num_elements & (num_elements - 1)) == 0
        ), "num_elements must be a power of 2"
        self.num_elements = num_elements
        self.tree = np.zeros(self.num_elements * 2)  # tree is rooted at idx 1, not 0
        self.size = 0

    def update(self, idx, value):
        #    print "sum_tree.update idx", idx, "value", value
        # note: we don't keep track of size and instead assume caller does.
        assert idx >=0 and idx < self.num_elements
        self.size = max(self.size, idx + 1)
        idx += self.num_elements  # shift into range of lower row
        delta = float(value) - self.tree[idx]
        while idx != 0:
            self.tree[idx] += delta
            idx >>= 1

    def value_at(self, idx):
        return self.tree[idx + self.num_elements]

    def total(self):
        return self.tree[1]

    def value_ntiles(self, n=11):
        current_values = self.tree[self.num_elements : self.num_elements + self.size]
        return np.percentile(
            current_values, np.linspace(0, 100, n), interpolation="nearest"
        )

    def index_of(self, value):
        if self.total() == 0:
            raise ValueError("no inserts yet?")
        assert value >= 0
        assert value <= self.total()
        idx = 1
        while True:
            peek_left = self.tree[idx << 1]
            if value >= peek_left:
                # continue down on rhs
                value -= peek_left
                idx = (idx << 1) + 1
            else:
                # continue down on lhs
                idx <<= 1
            # stop when we're in the final row
            if idx >= self.num_elements:
                return idx - self.num_elements

    def sample(self, n):
        # do sampling WITHOUT replacement by explicitly taking samples out of tree and
        # then returning them after all samples are taken. ( tried simpler rejection
        # sampling but wasn't as effective )
        #    print "sum_tree.sample"
        samples = []  # indexs and values
        for i in range(n):
            # sample and index and value
            idx = self.index_of(random.random() * self.total())
            value = self.value_at(idx)
            samples.append((idx, value))
            # remove temporarily from tree
            self.update(idx, 0)
        # replace entries in tree
        #    print "sum_tree.replace"
        for idx, value in samples:
            self.update(idx, value)
        # return samples
        #    print "sum_tree.sample returning", samples
        return samples

    def dump(self, additional_idx_data=None):
        print(">>dump")
        print("total", self.total())
        for idx, value in enumerate(self.tree):
            real_idx = idx - self.num_elements
            if real_idx < 0:
                real_idx = "."
                additional_data = "."
            elif additional_idx_data:
                additional_data = additional_idx_data[real_idx]
            print("\t".join(map(str, [idx, real_idx, additional_data, value])))
        print("<<dump")


In [ ]:
st = SumTree(num_elements=8)
st.dump()

for i in range(8):
    v = (i+1)*2.1
    print("adding", v, "at", i)
    st.update(i, (i+1)*2.1)
    st.dump()

In [ ]:


class ParametricCaptureImportanceSampledData(object):

    def __init__(
        self,
        run_root_dir: Path,
        unbiased_run_id: str,
        biased_run_ids: List[str],
        keras_model: str, # for losses
        alpha_huber: float=1.0,
        beta_stft: float=0.01,
        seed: int = 123,
        ignore_fade_len: int=500
    ):
        model_z_path = f"{run_root_dir}/{unbiased_run_id}/model_data.z"
        self.unbiased_data_z = zarr.open(model_z_path, mode="r")
        self.chunk_len = self.unbiased_data_z.chunks[0]
        print(unbiased_run_id, self.unbiased_data_z.nchunks)

        self.biased_data_z = []
        for run_id in biased_run_ids:
            model_z_path = f"{run_root_dir}/{run_id}/model_data.z"
            zd = zarr.open(model_z_path, mode="r")
            if zd.chunks[0] != self.chunk_len:
                raise Exception("biased data {d_z} has different chunk length than unbiased set")
            print(run_id, zd.nchunks)
            self.biased_data_z.append(zd)

        db = SampleDB()
        self.biased_records = []  # (zarr_idx, chunk_idx)
        self.biased_run_ids = biased_run_ids
        for i, run_id in enumerate(biased_run_ids):
            for loss_row in db.losses_for(run_id, model=keras_model):
                loss = alpha_huber * loss_row.huber + beta_stft * loss_row.stft
                print('run_id', run_id, 'idx', loss_row.idx, 'loss', loss)

        print('self.biased_records', self.biased_records)

        self.rng = np.random.default_rng(seed=seed)

    def sample_unbiased(self):
        r_chunk = self.rng.integers(low=0, high=self.n_chunks)
        r_seq_from = self.rng.integers(
            low=self.ignore_fade_len, high=self.chunk_len - IGNOignore_fade_lenRE_FADE_LEN - seq_len
        )
        r_seq_to = r_seq_from + seq_len
        # grab relevant pieces
        data = self.model_data_z.blocks[r_chunk][r_seq_from:r_seq_to]

In [ ]:
pcd = ParametricCaptureImportanceSampledData(
    run_root_dir = '../parametric_capture/runs',
    unbiased_run_id = '004',
    biased_run_ids = ['200', '201'],
    keras_model='230_keras/i0'
)

In [ ]:
hash(('300', 11, 0.0437))